# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1 — Ranked actions + reason codes
import os, getpass
import duckdb, pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('HF token: ')

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FEATURE_MONTHS = ["2026-01", "2026-02", "2026-03"]
feature_paths = [f"{REL}/fact_content_daily_performance/month={m}/data_0.parquet" for m in FEATURE_MONTHS]

data = con.sql(f"""
    SELECT
        content_hash_id, client_hash_id,
        SUM(gsc_impressions) AS impressions_janmar,
        SUM(gsc_clicks) AS clicks_janmar,
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS avg_position_janmar,
        SUM(ga4_sessions) AS sessions_janmar,
        SUM(scroll_events) AS scroll_events_janmar,
        SUM(CASE WHEN report_date >= DATE '2026-03-01' THEN gsc_impressions ELSE 0 END) AS impressions_march,
        MAX(CASE WHEN gsc_impressions > 0 THEN report_date END) AS last_observed_date
    FROM read_parquet({feature_paths})
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
    HAVING impressions_march >= 50
""").df()

data["days_since_last_activity"] = (pd.Timestamp("2026-04-01") - data["last_observed_date"]).dt.days

def staleness_score(days):
    if pd.isna(days): return 0, "NO_SIGNAL"
    if days >= 31: return 3, "RECENT_INACTIVITY_HIGH"
    if days >= 8:  return 2, "RECENT_INACTIVITY_MEDIUM"
    if days >= 3:  return 1, "RECENT_INACTIVITY_LOW"
    return 0, "RECENT_INACTIVITY_NONE"

data[["staleness_score", "staleness_reason"]] = data["days_since_last_activity"].apply(
    lambda d: pd.Series(staleness_score(d))
)

median_pos = data.loc[data["avg_position_janmar"] > 0, "avg_position_janmar"].median()
data["has_position_data"] = (data["avg_position_janmar"] > 0).astype(int)
data["avg_position_janmar"] = data["avg_position_janmar"].replace(0, median_pos)
for col in ["impressions_janmar", "clicks_janmar", "sessions_janmar", "scroll_events_janmar"]:
    data[f"log_{col}"] = np.log1p(data[col])

FEATURES_FINAL = ["log_impressions_janmar", "log_clicks_janmar", "avg_position_janmar",
                   "has_position_data", "log_sessions_janmar", "log_scroll_events_janmar"]

label_path = f"{REL}/fact_content_daily_performance/month=2026-04/data_0.parquet"
april = con.sql(f"""
    SELECT content_hash_id, client_hash_id, SUM(gsc_impressions) AS impressions_april
    FROM '{label_path}' WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()
data = data.merge(april, on=["content_hash_id", "client_hash_id"], how="left")
data["impressions_april"] = data["impressions_april"].fillna(0)
data["is_declining_label"] = (data["impressions_april"] < 0.8 * data["impressions_march"]).astype(int)

rf_final = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
                                   class_weight="balanced_subsample", random_state=42, n_jobs=-1)
rf_final.fit(data[FEATURES_FINAL], data["is_declining_label"])
data["model_probability"] = rf_final.predict_proba(data[FEATURES_FINAL])[:, 1]

data["staleness_norm"] = data["staleness_score"] / 3.0
data["final_score"] = 100 * (0.60 * data["model_probability"] + 0.40 * data["staleness_norm"])

def reason_codes(row):
    reasons = [row["staleness_reason"]]
    if row["model_probability"] >= 0.65: reasons.append("model_decline_risk")
    if row["impressions_march"] >= 500: reasons.append("high_visibility")
    return "|".join(reasons)

def suggested_action(row):
    if row["staleness_score"] >= 3 and row["model_probability"] >= 0.5: return "refresh_high_priority"
    if row["staleness_score"] >= 2 or row["model_probability"] >= 0.5: return "refresh"
    if row["staleness_score"] >= 1: return "review"
    return "monitor"

data["reason_codes"] = data.apply(reason_codes, axis=1)
data["suggested_action"] = data.apply(suggested_action, axis=1)
data = data.sort_values("final_score", ascending=False).reset_index(drop=True)
data["rank"] = data.index + 1

print(f"Rows scored: {len(data):,} | Base rate (is_declining_label): {data['is_declining_label'].mean():.3f}")
print(f"Clients represented in top 50: {data.head(50)['client_hash_id'].nunique()} of {data['client_hash_id'].nunique()} total")
data.head(20)[["rank","client_hash_id","final_score","model_probability","staleness_reason","suggested_action","reason_codes"]]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows scored: 116,114 | Base rate (is_declining_label): 0.518
Clients represented in top 50: 16 of 44 total


,rank,client_hash_id,final_score,model_probability,staleness_reason,suggested_action,reason_codes
0,1,client_08a6a72ff48e62c0,72.815229,0.769143,RECENT_INACTIVITY_MEDIUM,refresh,RECENT_INACTIVITY_MEDIUM|model_decline_risk
1,2,client_08a6a72ff48e62c0,71.618364,0.749195,RECENT_INACTIVITY_MEDIUM,refresh,RECENT_INACTIVITY_MEDIUM|model_decline_risk
2,3,client_08a6a72ff48e62c0,71.604168,0.748958,RECENT_INACTIVITY_MEDIUM,refresh,RECENT_INACTIVITY_MEDIUM|model_decline_risk
3,4,client_08a6a72ff48e62c0,70.625375,0.732645,RECENT_INACTIVITY_MEDIUM,refresh,RECENT_INACTIVITY_MEDIUM|model_decline_risk
4,5,client_08a6a72ff48e62c0,70.315114,0.727474,RECENT_INACTIVITY_MEDIUM,refresh,RECENT_INACTIVITY_MEDIUM|model_decline_risk
5,6,client_08a6a72ff48e62c0,69.560365,0.714895,RECENT_INACTIVITY_MEDIUM,refresh,RECENT_INACTIVITY_MEDIUM|model_decline_risk
6,7,client_08a6a72ff48e62c0,69.266734,0.710001,RECENT_INACTIVITY_MEDIUM,refresh,RECENT_INACTIVITY_MEDIUM|model_decline_risk
7,8,client_73cda7b4e4f265ea,68.692653,0.700433,RECENT_INACTIVITY_MEDIUM,refresh,RECENT_INACTIVITY_MEDIUM|model_decline_risk
8,9,client_08a6a72ff48e62c0,68.605115,0.698974,RECENT_INACTIVITY_MEDIUM,refresh,RECENT_INACTIVITY_MEDIUM|model_decline_risk
9,10,client_08a6a72ff48e62c0,68.553391,0.698112,RECENT_INACTIVITY_MEDIUM,refresh,RECENT_INACTIVITY_MEDIUM|model_decline_risk


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*


**What this supports:** a weekly reviewer queue — which pages a human should look at
first for possible refresh, based on staleness (recent inactivity) plus a weak
directional signal from a Jan-Mar → April decline model.

**Validated performance (from ML-09, client-grouped holdout — the honest split):**
ROC AUC 0.55-0.58, Average Precision 0.56-0.58, Precision@50 0.68-0.70. ROC AUC
this close to 0.50 means the model is only slightly better than random at
separating decliners from non-decliners. The base rate on this population is
~0.52, so Precision@50 of 0.68-0.70 is a real but modest lift over guessing.

**What this is NOT:** a prediction of what happens if nobody acts, a causal claim
that refreshing recovers traffic, or a general content-health score. It also does
not reuse the starter pipeline's 30k-row teaching numbers (ROC AUC 0.75) — that is
a different dataset and a different (same-window) label, not this capstone's
evidence.

**Known limits:**
- Single train/label window (Jan-Mar → April) — seasonality untested.
- Client-grouped holdout was small (9 clients) — treat the metric as directional.
- Staleness and model_probability partly disagree (see the sanity check above) —
  HIGH-staleness pages don't necessarily get the highest scores.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*


**Before acting on any row:** open the page, confirm it's genuinely stale (not
seasonal or a consolidation case per the lane guide), and check the reason codes
make sense in context.

**Never automate:**
- Auto-publishing, auto-archiving, or auto-deprioritizing from this score alone.
- Treating `model_decline_risk` as proof of a future decline — ROC AUC ~0.55
  means the model is frequently wrong.
- Batch-actioning many pages from one client without checking it's a real
  pattern, not a tie-breaking artifact.
- Reading `final_score` as a calibrated probability — it's an ordering device.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*


- Re-audit if the base rate on new data moves >10pp from 0.518.
- Re-audit if Precision@50 on a fresh holdout drops meaningfully below 0.68-0.70.
- Re-run the leakage test from w06 whenever a new feature is added.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, json
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

export_cols = ["rank","content_hash_id","client_hash_id","final_score",
               "model_probability","staleness_score","staleness_reason",
               "suggested_action","reason_codes","impressions_march",
               "days_since_last_activity"]
data[export_cols].to_csv("work/outputs/action_playbook_queue.csv", index=False)

metrics = {
    "label": "Jan-Mar features -> April decline (impressions_april < 0.8*impressions_march)",
    "features": FEATURES_FINAL,
    "validated_split": "client_grouped_holdout",
    "roc_auc": 0.553,
    "avg_precision": 0.562,
    "precision_at_50": 0.70,
    "base_rate": float(data["is_declining_label"].mean()),
    "rows_scored": int(len(data)),
    "source_notebooks": ["w05_model.ipynb", "w06_validation_audit.ipynb"],
}
with open("work/outputs/action_playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Wrote work/outputs/action_playbook_queue.csv and action_playbook_metrics.json")


Wrote work/outputs/action_playbook_queue.csv and action_playbook_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.